# Primul apel către un LLM din Colab

Cursul 1 · AI și programare

Rulează acest notebook în [Google Colab](https://colab.research.google.com/). Colab oferă modele Gemini gratuit prin modulul `google.colab.ai`, fără cont sau cheie API.

**Cum rulezi:** apasă pe celula de cod de mai jos, apoi `Shift + Enter` (sau butonul ▶).

In [ ]:
from google.colab import ai

intrebare = "Explică în română ce este căutarea binară, în 3 propoziții."
raspuns = ai.generate_text(intrebare)
print(raspuns)

## Încearcă singur

Schimbă textul din `intrebare` cu propria ta întrebare și rulează din nou celula. Observă cum răspunsul se modifică în funcție de ce ceri.

Idei:
- „Scrie o funcție Python care verifică dacă un număr este prim.”
- „Explică recursivitatea unui student la anul I, cu un exemplu.”

In [ ]:
intrebare = "Scrie o funcție Python care verifică dacă un număr este prim."
raspuns = ai.generate_text(intrebare)
print(raspuns)

# Rolurile mesajelor: `system`, `user`, `assistant`

Un apel către un model are, de obicei, trei tipuri de mesaje:

- **`system`** — instrucțiuni generale: cine este modelul și cum să răspundă (tonul, limba, formatul).
- **`user`** — întrebarea sau cererea concretă.
- **`assistant`** — răspunsul modelului, păstrat în istoricul conversației.

Modelul gratuit din Colab (`ai.generate_text`) primește un singur text, fără cheie API. Scriem explicit rolurile și le trimitem împreună. Întâi doar `system` + `user`, ca să vedem cum instrucțiunea `system` schimbă răspunsul.

In [ ]:
from google.colab import ai

system = "Ești un profesor de informatică. Răspunzi pe scurt, în română, cu un exemplu simplu."
user = "Ce este căutarea binară?"

# Combinăm rolul system (instrucțiuni) cu rolul user (întrebarea).
prompt = f"[system]\n{system}\n\n[user]\n{user}"
raspuns = ai.generate_text(prompt)
print(raspuns)

Schimbă doar variabila `system` (de exemplu: „Răspunde ca pentru un copil de 10 ani, fără termeni tehnici”) și rulează din nou, păstrând aceeași întrebare `user`. Vei vedea un răspuns diferit — asta arată rolul mesajului `system`.

## Rolul `assistant`: o conversație pe mai multe ture

`assistant` este răspunsul modelului, pe care îl păstrăm în **istoricul conversației**. Ca să continui discuția, retrimiți tot contextul: instrucțiunea `system`, întrebarea `user`, răspunsul `assistant` de dinainte și noua întrebare `user`.

Așa „ține minte” modelul ce s-a spus (vezi slide-urile 26 și 52 din curs).

In [ ]:
from google.colab import ai

system = "Ești un profesor de informatică. Răspunzi pe scurt, în română."

# Tura 1: system + user
user_1 = "Ce este căutarea binară?"
prompt_1 = f"[system]\n{system}\n\n[user]\n{user_1}"
assistant_1 = ai.generate_text(prompt_1)
print("Asistent (tura 1):\n", assistant_1)

# Tura 2: adăugăm răspunsul modelului (assistant) în istoric și punem o întrebare nouă.
user_2 = "Dă-mi un exemplu concret cu numere."
prompt_2 = (
    f"[system]\n{system}\n\n"
    f"[user]\n{user_1}\n\n"
    f"[assistant]\n{assistant_1}\n\n"
    f"[user]\n{user_2}"
)
assistant_2 = ai.generate_text(prompt_2)
print("\nAsistent (tura 2):\n", assistant_2)

Observă: la tura 2, modelul dă un exemplu care continuă explicația de la tura 1, fiindcă i-am inclus răspunsul anterior (`assistant`) în context. Dacă nu l-am fi inclus, ar fi răspuns „din senin”, fără legătură cu discuția.

> Notă: aici trimitem toată conversația ca un singur text, pentru că modelul gratuit din Colab nu primește roluri separate. În API-urile cu cheie (Gemini prin SDK, OpenAI etc.), `system`, `user` și `assistant` se trimit ca mesaje distincte, într-o listă.

# Tokeni: cum vede modelul textul

Un model nu vede cuvinte, ci **tokeni** — bucăți de text, fiecare cu un identificator numeric. Mai jos folosim biblioteca `tiktoken` ca să vedem cum se împarte un text.

Întâi instalăm biblioteca (o singură dată per sesiune Colab):

In [ ]:
!pip install -q tiktoken

In [ ]:
import tiktoken

text = "Ana învață Python și vrea să scrie un program care sortează o listă de numere."

# Împărțim textul în tokeni (identificatori numerici).
enc = tiktoken.get_encoding("o200k_base")
token_ids = enc.encode(text)

print(f"Text: {text}")
print(f"Număr de tokeni: {len(token_ids)}\n")

print("Nr. | ID token | Fragment")
print("----+----------+---------")
for i, token_id in enumerate(token_ids, 1):
    fragment = enc.decode([token_id])
    print(f"{i:3} | {token_id:8} | {fragment!r}")

Observă: „Python” este un singur token, dar „învață” se împarte în trei („ în”+„va”+„ță”). Tokenizatorul învață bucățile frecvente dintr-un corpus dominat de engleză, așa că textul în română ajunge să coste mai mulți tokeni.

Schimbă `text` cu o propoziție proprie și rulează din nou ca să vezi cum se împarte.